# Photo-z CNN — bins head (`arch='bins'`, Colab)
Post-bootcamp run: same trunk / data as v4, but the MDN head replaced by a **Pasquet-style
bin-classification head** (180 softmax bins over log1p(z), soft-label CE, point = E[z]).
Data = `catalog_v4` (cleaned 600k) + **sample_v4.5** (24×24×5 registered cutouts).

Baseline to beat: frozen v4 CNN + MDN head = **σMAD 0.01263** on the v4-5 val split.

Runtime → Change runtime type → **GPU**.


## 1. Setup + data
Clone the repo (code + `splits/`), install deps, log into Google, download catalog + shards (~3.5 GB).


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'        # sample_v4.5 is 24x24
![ -d /content/Photo-Z-CNN-Model ] || git clone -b main https://github.com/zhhrozhh/Photo-Z-CNN-Model.git /content/Photo-Z-CNN-Model
%cd /content/Photo-Z-CNN-Model
!pip -q install mlflow


In [ ]:
# Google auth — if this throws 'credential propagation was unsuccessful', just re-run this cell
# (allow the popup / pick the account that has access to macrocosm-lewagon)
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q


In [ ]:
!mkdir -p /content/data
# catalog_v4 (cleaned 600k, idx-aligned to sample_v4.5) -> saved as catalog_v1.parquet (the loader's expected name)
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/catalog_v1.parquet
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
DATA_DIR = '/content/data'

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))


## 2. MLflow token
Tracking server: `https://hangman-lab.top/ex-miscs/mlflow` (already the default `MLFLOW_URI` in `photoz_cnn.py`).
Add the token once under **Colab → 🔑 Secrets** as `MLFLOW_TOKEN` (value = content of `~/.mlflow_token` on the home machine).
No secret → trains without logging.


In [ ]:
try:
    from google.colab import userdata
    MLFLOW_TOKEN = userdata.get('MLFLOW_TOKEN')
except Exception as e:
    print('no MLFLOW_TOKEN secret ->', e); MLFLOW_TOKEN = None


## 3. Train (`arch='bins'`)
v4-5 splits (train 554,461 / val 45,357) — the same split every recent baseline uses, so σMAD is directly comparable.
TTA (8 dihedral views) on the final val pass.


In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir=DATA_DIR,
    crop=24,                 # sample_v4.5 is 24x24
    preproc='p99',
    arch='bins',             # bin-classification head
    bins=180, bins_smooth=1.0,
    train_csv='splits/v4-5-train.csv',
    val_csv='splits/v4-5-validate.csv',
    N=None,                  # cap #train images if RAM-limited; None = all (~554k)
    batch=256, lr=3e-4, epochs=60,
    tta=True,
    run_name='bins180-v1',
    experiment='photoz-cnn-bins',
    mlflow_token=MLFLOW_TOKEN,
)
print(metrics)


## 4. Save the model
MLflow already stores it as an artifact when logging is on; this cell is the manual backup path.


In [ ]:
model.save('/content/bins180-v1.keras')
# optional: push to GCS
# !gcloud storage cp /content/bins180-v1.keras gs://macrocosm-lewagon/models/
